# Comparación de líneas base · GPU T4
Subir la carpeta partes_v2 con Upload to Colab. Ejecutar todas las celdas. Las huellas verifican la reconstrucción de ambos ZIP.

In [ ]:
from pathlib import Path
import sys, hashlib, zipfile, json, csv
import torch
assert Path('/content').is_dir() and torch.cuda.is_available(), 'Se requiere un kernel GPU de Colab'
print(torch.cuda.get_device_name(0))
raiz=Path('/content/imo_v2')
salida=Path('/content/IMO_resultados_v2')
def extraer(nombre, huella):
    import time
    dst=Path('/content')/nombre
    if not dst.exists() or hashlib.sha256(dst.read_bytes()).hexdigest()!=huella:
        for intento in range(240):
            trozos=sorted(Path('/content').rglob(Path(nombre).stem+'_*.bin'))
            if sum(f.stat().st_size for f in trozos)=={'imo_colab_train_val.zip':184858955,'test_v2.zip':33081044}[nombre]:
                tmp=dst.with_suffix('.partial')
                with tmp.open('wb') as out:
                    for f in trozos: out.write(f.read_bytes())
                if hashlib.sha256(tmp.read_bytes()).hexdigest()==huella:
                    tmp.replace(dst)
                    break
            if intento % 6 == 0: print('Verificando transferencia:',nombre,len(trozos),flush=True)
            time.sleep(5)
        else: raise RuntimeError('Transferencia incompleta; verificar partes en /content')
    archivos=[p for p in {Path.cwd()/nombre,Path('/content')/nombre} if p.is_file()]
    assert archivos, f'Subir {nombre} con Upload to Colab'
    p=archivos[0]
    assert hashlib.sha256(p.read_bytes()).hexdigest()==huella, f'Paquete distinto: {nombre}'
    with zipfile.ZipFile(p) as z: z.extractall(raiz)
extraer('imo_colab_train_val.zip', '13045ae69c3cdfc60259d722624c258e57586737568df884e562fdacde42fedf')
scripts=raiz/'tesis_imo_para_codex/scripts'
sys.path.insert(0,str(scripts))


In [ ]:
codigo='"""Adaptaciones compactas para segmentación binaria, sin pesos externos."""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom modelos_base import block\n\nclass Piramide(nn.Module):\n    def __init__(self, modo):\n        super().__init__()\n        self.encoder = nn.Sequential(block(3,16), nn.MaxPool2d(2), block(16,32),\n                                     nn.MaxPool2d(2), block(32,64))\n        self.modo = modo\n        if modo == \'aspp\':\n            self.ramas = nn.ModuleList([nn.Conv2d(64,16,1)] + [\n                nn.Conv2d(64,16,3,padding=d,dilation=d) for d in (2,4,8)] +\n                [nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(64,16,1))])\n            canales = 80\n        else:\n            self.ramas = nn.ModuleList([nn.Sequential(nn.AdaptiveAvgPool2d(b),\n                           nn.Conv2d(64,16,1)) for b in (1,2,3,6)])\n            canales = 128\n        self.head = nn.Sequential(nn.Conv2d(canales,32,1),nn.ReLU(),nn.Conv2d(32,1,1))\n    def forward(self,x):\n        h = self.encoder(x)\n        partes = [h] if self.modo == \'psp\' else []\n        for rama in self.ramas:\n            y = F.relu(rama(h))\n            if y.shape[-2:] != h.shape[-2:]:\n                y = F.interpolate(y,size=h.shape[-2:],mode=\'bilinear\',align_corners=False)\n            partes.append(y)\n        return F.interpolate(self.head(torch.cat(partes,1)),size=x.shape[-2:],\n                             mode=\'bilinear\',align_corners=False)\nBASES = {\'deeplab_compacta\': lambda: Piramide(\'aspp\'), \'pspnet_compacta\': lambda: Piramide(\'psp\')}\n\n'
(scripts/'bases_v2.py').write_text(codigo, encoding='utf8')
from bases_v2 import BASES
from colab_runner import MODELS, run, evaluate, partition
MODELS.update(BASES)
seeds=[11,23,37,53,71]
identity=hashlib.sha256((raiz/'package_manifest.json').read_bytes()+codigo.encode()).hexdigest()
for semilla in seeds:
    for red in BASES:
        r=run(red,semilla,salida,epochs=30,lr=0.001,patience=5,identity=identity)
        assert r['completed']

In [ ]:
# Abrir el test reservado solo con las diez corridas cerradas.
for red in BASES:
    for s in seeds:
        assert json.loads((salida/f'{red}_seed{s}_lr0.001/result.json').read_text())['completed']
extraer('test_v2.zip', 'c324a4a69eed19f78413789a1d557f8a0880975462081ecf96926428b56c68a7')
prep=scripts.parent/'resultados/preentrenamiento'
freeze=json.loads((prep/'freeze_manifest.json').read_text())['artifacts']
for name in ['train.csv','val.csv','test.csv','normalization_train.json']:
    assert hashlib.sha256((prep/name).read_bytes()).hexdigest()==freeze[name]
rows=partition('test',allow_test=True)
assert len(rows)==218 and len({r['group_id'] for r in rows})==122
assert not {r['group_id'] for r in rows} & {r['group_id'] for part in ['train','val'] for r in partition(part)}
detalle=[]
for red in BASES:
    for s in seeds:
        net=BASES[red]().cuda()
        ck=(salida/f'{red}_seed{s}_lr0.001/best.pt')
        net.load_state_dict(torch.load(ck,map_location='cuda',weights_only=True))
        score,detail=evaluate(net,rows,torch.device('cuda'))
        detalle.extend(dict(model=red,seed=s,**r) for r in detail)
        print(red,s,score)
with (salida/'por_imagen.csv').open('w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=detalle[0]); w.writeheader(); w.writerows(detalle)
import shutil
print(shutil.make_archive('/content/IMO_resultados_v2','zip',salida))
